In [1]:
import os
import rasterio
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
import imageio
import imageio.v3 as iio  # For creating GIFs
import rasterio
from matplotlib import rcParams
import matplotlib.patches as patches
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from shapely.geometry import LineString
import geopandas as gpd
from scipy.interpolate import UnivariateSpline
import contextily as ctx
import plotly.express as px  # For alluvial (Sankey) plots
from shapely.geometry import Polygon, MultiPolygon
from rasterio.mask import mask
import pandas as pd
import geopandas as gpd
from shapely.geometry import mapping
from collections import Counter
from shapely.ops import split
from matplotlib.transforms import Affine2D
from matplotlib.patches import Rectangle
from rasterio.plot import show

In [2]:
#### Generating the segmented subregion polygons #########

# Paths
polygon_path = "/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_reprojected_revise.shp"
transects_path = "/Volumes/MaloneLab/Research/Whitezone/Spatial_Data/transects_37_61.shp"
#output_path = "/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_split_2.shp"

#transects_path = "/Volumes/MaloneLab/Research/Whitezone/Spatial_Data/line_ID_39.shp"
#transects_path = "/Volumes/MaloneLab/Research/Whitezone/Spatial_Data/line_ID_39.shp"
output_path = "/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_split_new.shp"
# Load data
polygon_gdf = gpd.read_file(polygon_path)
transects_gdf = gpd.read_file(transects_path)

# Ensure CRS match
if polygon_gdf.crs != transects_gdf.crs:
    transects_gdf = transects_gdf.to_crs(polygon_gdf.crs)

# Start with original polygon geometries
current_geoms = list(polygon_gdf.geometry)

# Loop through each transect (LineString) and split polygons one at a time
for line in transects_gdf.geometry:
    new_geoms = []
    for geom in current_geoms:
        if geom.intersects(line):
            result = split(geom, line)
            new_geoms.extend(result.geoms)
        else:
            new_geoms.append(geom)
    current_geoms = new_geoms  # Update for next line

# Create GeoDataFrame from final split geometries
split_gdf = gpd.GeoDataFrame(geometry=current_geoms, crs=polygon_gdf.crs)

# Save to shapefile
split_gdf.to_file(output_path)

print(f"✅ Saved split polygons to: {output_path}")


✅ Saved split polygons to: /Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_split_new.shp


In [4]:
### Output rasters for each subregion 

# Paths
raster_folder = '/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/Roy'
shapefile_path = '/Users/bz294/Documents/WhiteZone/Points/Shapefiles/SEregion_split_new.shp'
base_output_folder = os.path.join(raster_folder, 'Subregion_new')

# Load polygons
subregions = gpd.read_file(shapefile_path)

# Define remapping: original index -> desired region folder index
# index 0 → Region_2
# index 1 → Region_1
# index 2 → Region_0
# index 3 → Region_3
index_map = {0: 2, 1: 1, 2: 0, 3: 3}

# Create remapped region folders
region_folders = {}
for orig_idx, target_idx in index_map.items():
    folder = os.path.join(base_output_folder, f"Region_{target_idx}")
    os.makedirs(folder, exist_ok=True)
    region_folders[orig_idx] = folder

# Get raster paths
raster_paths = sorted([
    os.path.join(raster_folder, f)
    for f in os.listdir(raster_folder)
    if f.endswith('.tif') and f.startswith('raster')
])

# Loop over rasters
for raster_path in raster_paths:
    raster_name = os.path.splitext(os.path.basename(raster_path))[0]
    
    with rasterio.open(raster_path) as src:
        for i, polygon in enumerate(subregions.geometry):
            try:
                out_image, out_transform = mask(src, [mapping(polygon)], crop=True)
                out_meta = src.meta.copy()
                out_meta.update({
                    "driver": "GTiff",
                    "height": out_image.shape[1],
                    "width": out_image.shape[2],
                    "transform": out_transform
                })

                # Save to remapped folder
                out_path = os.path.join(region_folders[i], f"{raster_name}.tif")
                with rasterio.open(out_path, "w", **out_meta) as dest:
                    dest.write(out_image)
            except Exception as e:
                print(f"⚠️ Could not mask {raster_name} with subregion {i}: {e}")

print("✅ Rasters saved with remapped region folders (0→2, 2→0).")


✅ Rasters saved with remapped region folders (0→2, 2→0).
